In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Reload features from Delta
sales_final = spark.table("workspace.default.m5_features")

print(f"Loaded: {sales_final.count():,} rows, {len(sales_final.columns)} columns")
sales_final.printSchema()

In [0]:
# Chronological split - last 28 days as test
SPLIT_DATE = "2016-04-25"

train_df = sales_final.filter(F.col("date") < SPLIT_DATE)
test_df = sales_final.filter(F.col("date") >= SPLIT_DATE)

# Normally counts are not necessary and can cause perfomance issue, but using them to make sure splits are correct
train_count = train_df.count()
test_count = test_df.count()

print(f"Train: {train_count:,} rows  ({train_df.agg(F.min('date')).collect()[0][0]} to {train_df.agg(F.max('date')).collect()[0][0]})")
print(f"Test:  {test_count:,} rows  ({test_df.agg(F.min('date')).collect()[0][0]} to {test_df.agg(F.max('date')).collect()[0][0]})")
print(f"Test ratio: {100 * test_count / (train_count + test_count):.1f}%")

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Categorical columns to encode
categorical_cols = ["cat_id", "dept_id", "state_id", "store_id"]

# Step 1: StringIndexer - string -> numeric index
indexers = [
    StringIndexer(
        inputCol=c, 
        outputCol=f"{c}_idx", 
        handleInvalid="keep"  # unseen categories in test -> special bucket
    )
    for c in categorical_cols
]

# Step 2: OneHotEncoder - index -> sparse one-hot vector
encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx", 
        outputCol=f"{c}_ohe"
    )
    for c in categorical_cols
]

print("Indexers and encoders defined:")
for c in categorical_cols:
    print(f"  {c} -> {c}_idx -> {c}_ohe")

In [0]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
import time

# Recent 2-year window - addresses distribution shift + much faster
RECENT_START = "2014-04-25"

train_recent = sales_final.filter(
    (F.col("date") >= RECENT_START) & (F.col("date") < SPLIT_DATE)
)

print(f"Recent train rows: {train_recent.count():,}")

# Model - can afford 20 trees now that data is smaller
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="sales",
    maxIter=20,
    maxDepth=6,
    stepSize=0.1,
    seed=42,
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, gbt])

print("Training on recent 2-year window (20 trees)...")
start = time.time()

model = pipeline.fit(train_recent)

elapsed = time.time() - start
print(f"Training complete in {elapsed/60:.1f} minutes")

In [0]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
import time

# Recent 2-year window - addresses distribution shift + much faster
RECENT_START = "2014-04-25"

train_recent = sales_final.filter(
    (F.col("date") >= RECENT_START) & (F.col("date") < SPLIT_DATE)
)

print(f"Recent train rows: {train_recent.count():,}")

# Model - can afford 20 trees now that data is smaller
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="sales",
    maxIter=20,
    maxDepth=6,
    stepSize=0.1,
    seed=42,
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, gbt])

print("Training on recent 2-year window (20 trees)...")
start = time.time()

model = pipeline.fit(train_recent)

elapsed = time.time() - start
print(f"Training complete in {elapsed/60:.1f} minutes")

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# Predict on the test set (last 28 days)
predictions = model.transform(test_df)

# Evaluate
rmse_eval = RegressionEvaluator(
    labelCol="sales", predictionCol="prediction", metricName="rmse"
)
mae_eval = RegressionEvaluator(
    labelCol="sales", predictionCol="prediction", metricName="mae"
)

rmse = rmse_eval.evaluate(predictions)
mae = mae_eval.evaluate(predictions)

print(f"GBT Model — Test RMSE: {rmse:.4f}")
print(f"GBT Model — Test MAE:  {mae:.4f}")

# Peek at some predictions vs actuals
predictions.select("item_id", "store_id", "date", "sales", "prediction") \
    .filter(F.col("sales") > 0) \
    .show(10)

In [0]:
# Naive baseline: predict sales = sales_lag_7 (last week's same day)
baseline = test_df.withColumn("prediction", F.col("sales_lag_7").cast("double"))

# Drop rows where lag_7 is null (can't predict)
baseline = baseline.filter(F.col("prediction").isNotNull())

baseline_rmse = rmse_eval.evaluate(baseline)
baseline_mae = mae_eval.evaluate(baseline)

print(f"Naive baseline (lag_7) — RMSE: {baseline_rmse:.4f}")
print(f"Naive baseline (lag_7) — MAE:  {baseline_mae:.4f}")
print()
print(f"GBT model              — RMSE: {rmse:.4f}")
print(f"GBT model              — MAE:  {mae:.4f}")
print()
print(f"RMSE improvement: {100*(baseline_rmse - rmse)/baseline_rmse:.1f}%")
print(f"MAE improvement:  {100*(baseline_mae - mae)/baseline_mae:.1f}%")

In [0]:
# Extract the trained GBT model (last stage of the pipeline)
gbt_model = model.stages[-1]

# Get feature importances
importances = gbt_model.featureImportances

# We need feature names - reconstruct from assembler input
# Numeric features keep their names; OHE features expand
feature_names = numeric_cols.copy()
for c in categorical_cols:
    # Each OHE column expands to (num_categories - 1) slots
    # We'll label them generically
    n_cats = model.stages[categorical_cols.index(c)].labelsArray[0]
    for i in range(len(n_cats) - 1):
        feature_names.append(f"{c}={n_cats[i]}")

# Pair names with importances and sort
importance_list = [
    (feature_names[i], float(importances[i])) 
    for i in range(len(feature_names)) 
    if i < len(importances)
]
importance_list.sort(key=lambda x: x[1], reverse=True)

print("Top 15 features by importance:")
for name, imp in importance_list[:15]:
    bar = "█" * int(imp * 100)
    print(f"  {name:<28} {imp:.4f}  {bar}")

In [0]:
import matplotlib.pyplot as plt

# Take top 12 features from the importance_list we already computed
top_features = importance_list[:12]
names = [x[0] for x in top_features][::-1]   # reverse for horizontal bar
values = [x[1] for x in top_features][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, values, color="#4C72B0")
ax.set_xlabel("Feature Importance")
ax.set_title("GBT Feature Importance — Top 12", fontsize=13, fontweight="bold")

# Add value labels on bars
for bar, val in zip(bars, values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, 
            f"{val:.3f}", va="center", fontsize=9)

ax.set_xlim(0, max(values) * 1.15)
plt.tight_layout()
plt.savefig("/tmp/fig1_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved fig1_feature_importance.png")

In [0]:
import matplotlib.pyplot as plt

# Pick a few representative item-store combos that have real sales
sample_series = predictions.filter(
    (F.col("item_id").isin(["FOODS_3_090", "FOODS_3_586", "HOUSEHOLD_1_118"])) &
    (F.col("store_id") == "CA_1")
).select("item_id", "date", "sales", "prediction").toPandas()

# Sort by date within each series
sample_series = sample_series.sort_values(["item_id", "date"])

# Plot: one subplot per item
items = sample_series["item_id"].unique()
fig, axes = plt.subplots(len(items), 1, figsize=(12, 3*len(items)), sharex=True)
if len(items) == 1:
    axes = [axes]

for ax, item in zip(axes, items):
    d = sample_series[sample_series["item_id"] == item]
    ax.plot(d["date"], d["sales"], marker="o", label="Actual", color="#2C3E50", linewidth=2)

In [0]:
# Check what's actually in the sample
check = predictions.filter(
    (F.col("item_id").isin(["FOODS_3_090", "FOODS_3_586", "HOUSEHOLD_1_118"])) &
    (F.col("store_id") == "CA_1")
).select("item_id", "date", "sales", "prediction")

check.show(10)
print(f"Row count: {check.count()}")
print(f"Null predictions: {check.filter(F.col('prediction').isNull()).count()}")

In [0]:
import matplotlib.pyplot as plt

# Reload the sample (fresh)
sample_series = predictions.filter(
    (F.col("item_id").isin(["FOODS_3_090", "FOODS_3_586", "HOUSEHOLD_1_118"])) &
    (F.col("store_id") == "CA_1")
).select("item_id", "date", "sales", "prediction").toPandas()

sample_series = sample_series.sort_values(["item_id", "date"])
items = sorted(sample_series["item_id"].unique())

# Create fresh figure
plt.close("all")
fig, axes = plt.subplots(len(items), 1, figsize=(12, 3.2*len(items)))

for i, item in enumerate(items):
    ax = axes[i]
    d = sample_series[sample_series["item_id"] == item].sort_values("date")
    ax.plot(d["date"].values, d["sales"].values, marker="o", label="Actual",
            color="#2C3E50", linewidth=2, markersize=5)
    ax.plot(d["date"].values, d["prediction"].values, marker="s", label="Predicted",
            color="#E74C3C", linewidth=2, markersize=5, alpha=0.85)
    ax.set_title(f"{item} @ CA_1", fontsize=11, fontweight="bold")
    ax.set_ylabel("Units sold")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date")
fig.suptitle("Forecast vs Actual — Test Period (28 days)",
             fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.98])
fig.savefig("/tmp/fig2_forecast_vs_actual.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved fig2_forecast_vs_actual.png")

In [0]:
import matplotlib.pyplot as plt

# Aggregate daily totals: actual vs predicted across ALL products
daily_agg = predictions.groupBy("date").agg(
    F.sum("sales").alias("actual_total"),
    F.sum("prediction").alias("predicted_total")
).orderBy("date").toPandas()

plt.close("all")
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily_agg["date"].values, daily_agg["actual_total"].values,
        marker="o", label="Actual", color="#2C3E50", linewidth=2.5, markersize=6)
ax.plot(daily_agg["date"].values, daily_agg["predicted_total"].values,
        marker="s", label="Predicted", color="#E74C3C", linewidth=2.5, markersize=6, alpha=0.85)
ax.set_title("Aggregate Daily Sales — Actual vs Predicted (All Products, Test Period)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Total units sold")
ax.legend(loc="upper right", fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("/tmp/fig3_aggregate.png", dpi=120, bbox_inches="tight")
plt.show()

# Compute aggregate-level error
daily_agg["error_pct"] = 100 * (daily_agg["predicted_total"] - daily_agg["actual_total"]) / daily_agg["actual_total"]
print(f"Mean aggregate error: {daily_agg['error_pct'].mean():.1f}%")
print(f"Mean absolute aggregate error: {daily_agg['error_pct'].abs().mean():.1f}%")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Sample residuals (error = prediction - actual) for products with real sales
resid_sample = predictions.filter(F.col("sales") > 0) \
    .withColumn("residual", F.col("prediction") - F.col("sales")) \
    .select("residual") \
    .sample(fraction=0.02, seed=42) \
    .toPandas()

plt.close("all")
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(resid_sample["residual"].values, bins=60, color="#4C72B0",
        edgecolor="white", alpha=0.8)
ax.axvline(0, color="#2C3E50", linestyle="--", linewidth=2, label="Perfect prediction")
ax.axvline(resid_sample["residual"].mean(), color="#E74C3C", linestyle="-",
           linewidth=2, label=f"Mean residual = {resid_sample['residual'].mean():.2f}")
ax.set_title("Residual Distribution (Prediction − Actual, non-zero sales)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Residual (units)")
ax.set_ylabel("Frequency")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(-15, 15)
fig.tight_layout()
fig.savefig("/tmp/fig4_residuals.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Mean residual: {resid_sample['residual'].mean():.3f}")
print(f"Median residual: {resid_sample['residual'].median():.3f}")

In [0]:
import os

# Check where the git folder is in the workspace
# List /tmp to confirm our figures are there
print("Figures in /tmp:")
for f in os.listdir("/tmp"):
    if f.endswith(".png"):
        path = os.path.join("/tmp", f)
        size = os.path.getsize(path) / 1024
        print(f"  {f}  ({size:.0f} KB)")

In [0]:
import shutil

# Copy figures from /tmp to our volume (which we can browse/download)
VOLUME_FIGURES = "/Volumes/workspace/default/m5_forecasting/figures"

# Create the figures directory
dbutils.fs.mkdirs(VOLUME_FIGURES)

figures = [
    "fig1_feature_importance.png",
    "fig2_forecast_vs_actual.png",
    "fig3_aggregate.png",
    "fig4_residuals.png",
]

for fig in figures:
    src = f"/tmp/{fig}"
    dst = f"{VOLUME_FIGURES}/{fig}"
    shutil.copy(src, dst)
    print(f"Copied {fig}")

print("\nAll figures copied to volume.")